# Portable Blood Cell Classification Pipeline

## GitHub-first Colab/Kaggle workflow

This notebook downloads the project code from `https://github.com/Sushey01/DeepLearning` and the TXL-PBC dataset from `https://github.com/lugan113/TXL-PBC_Dataset`. It does not depend on the author's Windows folder.

Enable a GPU runtime in Colab or Kaggle, then run the cells from top to bottom. The notebook normalizes the dataset layout, applies deduplication to a writable copy, trains EfficientNet-B0, MobileNetV3-Small, and DenseNet121, evaluates all three models, and generates the comparison outputs.

Use `DATA_ROOT_OVERRIDE` only when the dataset is already mounted or uploaded. Use `USE_GITHUB_SOURCE = False` and `PROJECT_ROOT_OVERRIDE` only when using local project code.

In [10]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile
from urllib.request import urlopen

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle").exists()
MODELS = ["efficientnet_b0", "mobilenet_v3_small", "densenet121"]

# GitHub is the default source so this notebook is independent of the author's computer.
USE_GITHUB_SOURCE = True
GITHUB_SOURCE_URL = "https://github.com/Sushey01/DeepLearning/archive/refs/heads/main.zip"
DATASET_SOURCE_URL = "https://github.com/lugan113/TXL-PBC_Dataset/archive/refs/heads/master.zip"

# Set USE_GITHUB_SOURCE=False to use a local or uploaded project source instead.
PROJECT_ROOT_OVERRIDE = ""
# Set this only when the dataset is already downloaded elsewhere.
DATA_ROOT_OVERRIDE = ""

# Keep True for the cleaned TXL-PBC dataset used by this project.
ASSERT_EXPECTED_COUNTS = True
EXPECTED_TEST_COUNTS = {"WBC": 127, "RBC": 1593, "Platelet": 48}

if IN_KAGGLE:
    RUN_ROOT = Path("/kaggle/working/blood_cell_classification_run")
elif IN_COLAB:
    RUN_ROOT = Path("/content/blood_cell_classification_run")
else:
    RUN_ROOT = Path.cwd() / "portable_blood_cell_classification_run"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({"colab": IN_COLAB, "kaggle": IN_KAGGLE, "run_root": str(RUN_ROOT)})

{'colab': False, 'kaggle': False, 'run_root': 'C:\\Users\\MSI\\Shekhar\\DeepLearning\\portable_blood_cell_classification_run'}


## Install dependencies

The cell deliberately does not reinstall PyTorch or torchvision. Kaggle and Colab normally provide those packages with GPU support.

In [11]:
required = [
    "numpy", "pandas", "scikit-learn", "matplotlib",
    "seaborn", "pyyaml", "pillow", "imagehash", "tqdm"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *required], check=True)
import torch
import torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Warning: training will run on CPU and may take a long time.")

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
CUDA available: True


## Locate source code and data

The notebook downloads both remote dependencies by default:

- Project code: `https://github.com/Sushey01/DeepLearning`
- TXL-PBC dataset: `https://github.com/lugan113/TXL-PBC_Dataset`

The TXL-PBC repository stores images and labels in split folders. The notebook automatically converts that layout into the flat `data/raw/images`, `data/raw/labels`, and split-list layout expected by this project.

For a local or pre-downloaded dataset, set `DATA_ROOT_OVERRIDE` to a folder containing `processed/` or `raw/`. Set `USE_GITHUB_SOURCE = False` and `PROJECT_ROOT_OVERRIDE` only when using local project code. Input data is copied into a writable runtime directory before deduplication.

In [ ]:
def find_project_root(starts):
    for start in starts:
        start = Path(start).expanduser()
        if not start.exists():
            continue

        candidates = [start, *start.parents]
        candidates.extend(path.parent.parent for path in start.rglob("train.py"))
        for candidate in candidates:
            if (candidate / "src" / "train.py").exists() and (candidate / "src" / "evaluate.py").exists():
                return candidate.resolve()
    return None


def normalize_data_root(path):
    path = Path(path).expanduser()
    if (path / "processed").exists() or (path / "raw").exists():
        return path.resolve()
    if path.name in {"processed", "raw"} and path.exists():
        return path.parent.resolve()
    return path.resolve()


def find_txlpbc_root(search_root):
    for images_dir in Path(search_root).rglob("images"):
        candidate = images_dir.parent
        if (candidate / "images" / "train").exists() and (candidate / "labels" / "train").exists():
            return candidate
    return None


def download_txlpbc_as_project_data():
    archive_path = RUN_ROOT / "txl_pbc_dataset.zip"
    extract_root = RUN_ROOT / "downloaded_txl_pbc"
    if not archive_path.exists():
        print("Downloading TXL-PBC dataset from GitHub...")
        with urlopen(DATASET_SOURCE_URL) as response, open(archive_path, "wb") as output:
            shutil.copyfileobj(response, output)
    if not extract_root.exists():
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_root)

    dataset_root = find_txlpbc_root(extract_root)
    if dataset_root is None:
        raise FileNotFoundError("Downloaded TXL-PBC archive does not contain images/{train,val,test} and labels/{train,val,test}.")

    normalized_root = RUN_ROOT / "downloaded_dataset_data"
    if normalized_root.exists():
        shutil.rmtree(normalized_root)
    raw_images = normalized_root / "raw" / "images"
    raw_labels = normalized_root / "raw" / "labels"
    raw_images.mkdir(parents=True, exist_ok=True)
    raw_labels.mkdir(parents=True, exist_ok=True)

    for split in ("train", "val", "test"):
        split_images = sorted((dataset_root / "images" / split).glob("*"))
        split_labels = sorted((dataset_root / "labels" / split).glob("*.txt"))
        for image_path in split_images:
            if image_path.is_file():
                shutil.copy2(image_path, raw_images / image_path.name)
        for label_path in split_labels:
            shutil.copy2(label_path, raw_labels / label_path.name)
        (normalized_root / "raw" / f"{split}.txt").write_text(
            "\n".join(path.name for path in split_images if path.is_file()) + "\n",
            encoding="utf-8",
        )

    print("Normalized TXL-PBC dataset:", normalized_root)
    return normalized_root

PROJECT_ROOT = None
if not USE_GITHUB_SOURCE:
    project_starts = [Path.cwd(), RUN_ROOT]
    if PROJECT_ROOT_OVERRIDE:
        project_starts.insert(0, Path(PROJECT_ROOT_OVERRIDE))
    PROJECT_ROOT = find_project_root(project_starts)

if PROJECT_ROOT is None:
    archive_path = RUN_ROOT / "project.zip"
    extract_root = RUN_ROOT / "downloaded_project"
    if not archive_path.exists():
        print("Downloading project source from GitHub...")
        with urlopen(GITHUB_SOURCE_URL) as response, open(archive_path, "wb") as output:
            shutil.copyfileobj(response, output)
    if not extract_root.exists():
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_root)
    PROJECT_ROOT = find_project_root([extract_root])

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project source could not be found after the GitHub download. "
        "Check internet access and confirm the repository contains src/train.py and src/evaluate.py. "
        "For local execution, set USE_GITHUB_SOURCE=False and PROJECT_ROOT_OVERRIDE."
    )

data_candidates = []
if DATA_ROOT_OVERRIDE:
    data_candidates.append(normalize_data_root(DATA_ROOT_OVERRIDE))
if not USE_GITHUB_SOURCE:
    data_candidates.extend([PROJECT_ROOT / "data", Path.cwd() / "data"])
data_candidates.extend([
    Path("/kaggle/input/blood-cell-data/data"),
    Path("/kaggle/input/blood-cell-classification/data"),
    Path("/content/drive/MyDrive/blood-cell-classification/data"),
    Path("/content/drive/MyDrive/data"),
])
DATA_SOURCE = next(
    (path for path in data_candidates if (path / "processed").exists() or (path / "raw").exists()),
    None,
)
if DATA_SOURCE is None:
    DATA_SOURCE = download_txlpbc_as_project_data()

RUNTIME_DATA = RUN_ROOT / "data"
if RUNTIME_DATA.exists():
    shutil.rmtree(RUNTIME_DATA)
shutil.copytree(DATA_SOURCE, RUNTIME_DATA)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_ROOT)
print("Project source:", PROJECT_ROOT)
print("Dataset source:", DATA_SOURCE)
print("Writable dataset copy:", RUNTIME_DATA)

Normalized TXL-PBC dataset: C:\Users\MSI\Shekhar\DeepLearning\portable_blood_cell_classification_run\downloaded_dataset_data


KeyboardInterrupt: 

In [ ]:
import yaml

OUTPUT_ROOT = RUN_ROOT / "outputs_post_dedup"
CONFIG_PATH = RUN_ROOT / "config_post_dedup.yaml"
config = {
    "raw_images_dir": str(RUNTIME_DATA / "raw" / "images"),
    "raw_labels_dir": str(RUNTIME_DATA / "raw" / "labels"),
    "split_files": {
        "train": str(RUNTIME_DATA / "raw" / "train.txt"),
        "val": str(RUNTIME_DATA / "raw" / "val.txt"),
        "test": str(RUNTIME_DATA / "raw" / "test.txt"),
    },
    "processed_dir": str(RUNTIME_DATA / "processed"),
    "checkpoints_dir": str(OUTPUT_ROOT / "checkpoints"),
    "logs_dir": str(OUTPUT_ROOT / "logs"),
    "results_dir": str(OUTPUT_ROOT / "results"),
    "classes": ["WBC", "RBC", "Platelet"],
    "image_size": 224,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],
    "augment": {"rotation_degrees": 20, "horizontal_flip_prob": 0.5, "brightness_jitter": 0.2, "contrast_jitter": 0.2, "minority_classes": ["WBC", "Platelet"], "minority_oversample_factor": 3},
    "batch_size": 32,
    "num_epochs": 30,
    "learning_rate": 0.0003,
    "weight_decay": 0.00001,
    "early_stopping_patience": 6,
    "loss_type": "weighted_ce",
    "focal_gamma": 2.0,
    "effective_number_beta": 0.999,
    "models": MODELS,
    "seed": 42,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
cfg = config
print("Config:", CONFIG_PATH)
print("Output directory:", OUTPUT_ROOT)

Config: c:\Users\MSI\Shekhar\DeepLearning\portable_blood_cell_classification_run\config_post_dedup.yaml
Output directory: c:\Users\MSI\Shekhar\DeepLearning\portable_blood_cell_classification_run\outputs_post_dedup


## Prepare, deduplicate, and verify the dataset

If only raw data is supplied, crops are created first. Deduplication is applied to the writable runtime copy, never to the original Kaggle or Drive input.

In [ ]:
import crop_dataset
import deduplicate_processed_dataset
from utils import count_images_per_class

processed_dir = Path(cfg["processed_dir"])
if not processed_dir.exists() or not any(processed_dir.rglob("*.png")):
    raw_images = Path(cfg["raw_images_dir"])
    raw_labels = Path(cfg["raw_labels_dir"])
    if not raw_images.exists() or not raw_labels.exists():
        raise FileNotFoundError("Processed data is missing and raw images/labels were not supplied.")
    crop_dataset.main(str(CONFIG_PATH))

before_counts = {}
for split in ("train", "val", "test"):
    before_counts[split] = count_images_per_class(str(processed_dir / split), cfg["classes"])
print("Counts before cleanup:", before_counts)

deduplication_processed = deduplicate_processed_dataset.deduplicate_processed_dataset(apply_cleanup=True)

after_counts = {}
for split in ("train", "val", "test"):
    after_counts[split] = count_images_per_class(str(processed_dir / split), cfg["classes"])
print("Counts after cleanup:", after_counts)

if ASSERT_EXPECTED_COUNTS:
    actual_test_counts = after_counts["test"]
    if actual_test_counts != EXPECTED_TEST_COUNTS:
        raise AssertionError(
            f"Unexpected cleaned test counts: {actual_test_counts}; "
            f"expected {EXPECTED_TEST_COUNTS}. "
            "Set ASSERT_EXPECTED_COUNTS=False for another dataset version."
        )
print("Verified cleaned test total:", sum(after_counts["test"].values()))

Counts before cleanup: {'train': {'WBC': 901, 'RBC': 10854, 'Platelet': 382}, 'val': {'WBC': 252, 'RBC': 3180, 'Platelet': 112}, 'test': {'WBC': 127, 'RBC': 1593, 'Platelet': 48}}
{
  "before": {
    "train": {
      "WBC": 901,
      "RBC": 10854,
      "Platelet": 382
    },
    "val": {
      "WBC": 252,
      "RBC": 3180,
      "Platelet": 112
    },
    "test": {
      "WBC": 127,
      "RBC": 1593,
      "Platelet": 48
    }
  },
  "after": {
    "train": {
      "WBC": 901,
      "RBC": 10854,
      "Platelet": 382
    },
    "val": {
      "WBC": 252,
      "RBC": 3180,
      "Platelet": 112
    },
    "test": {
      "WBC": 127,
      "RBC": 1593,
      "Platelet": 48
    }
  },
  "removed_count": 0,
  "removed_by_class": {
    "WBC": 0,
    "RBC": 0,
    "Platelet": 0
  },
  "pairwise_counts": {
    "train_val": {
      "WBC": 0,
      "RBC": 0,
      "Platelet": 0
    },
    "train_test": {
      "WBC": 0,
      "RBC": 0,
      "Platelet": 0
    },
    "val_test": {
      "W

## Train all three models from scratch

This always writes new checkpoints and logs under `outputs_post_dedup/`; it never reuses old checkpoints. ImageNet weights may be downloaded by torchvision on the first model build.

In [ ]:
import train

training_times = {}
for model_name in MODELS:
    start = time.perf_counter()
    train.main(str(CONFIG_PATH), model_name)
    training_times[model_name] = time.perf_counter() - start
    print(f"{model_name} training wall time: {training_times[model_name]:.2f} seconds")

Training class counts: {'WBC': 901, 'RBC': 10854, 'Platelet': 382}
efficientnet_b0: 4,011,391 trainable params


Training efficientnet_b0:   3%|▎         | 1/30 [01:50<53:19, 110.33s/it]

Epoch 1: train_loss=0.0421 train_f1=0.9853 val_loss=0.0102 val_f1=0.9970 time=110.3s mem=2834MB


Training efficientnet_b0:   7%|▋         | 2/30 [03:14<44:12, 94.75s/it] 

Epoch 2: train_loss=0.0137 train_f1=0.9955 val_loss=0.0101 val_f1=0.9955 time=194.2s mem=2834MB


...

Epoch 4: train_loss=0.0090 train_f1=0.9968 val_loss=0.0191 val_f1=0.9949 time=355.6s mem=2834MB


Training efficientnet_b0:  17%|█▋        | 5/30 [07:14<34:34, 82.99s/it]

Epoch 5: train_loss=0.0037 train_f1=0.9986 val_loss=0.0139 val_f1=0.9964 time=434.6s mem=2834MB


Training efficientnet_b0:  17%|█▋        | 5/30 [07:18<36:30, 87.60s/it]


KeyboardInterrupt: 

## Evaluate all models and generate figures

In [ ]:
import evaluate
import plot_model_results

evaluation_times = {}
for model_name in MODELS:
    start = time.perf_counter()
    evaluate.main(str(CONFIG_PATH), model_name)
    evaluation_times[model_name] = time.perf_counter() - start
    print(f"{model_name} evaluation wall time: {evaluation_times[model_name]:.2f} seconds")

evaluate.main(str(CONFIG_PATH), "all")
plot_model_results.plot_loss_curves(str(CONFIG_PATH))
print("Generated comparison table, confusion matrices, and loss curves in", cfg["results_dir"])

In [ ]:
import json

results_dir = Path(cfg["results_dir"])
comparison = json.loads((results_dir / "comparison_table.json").read_text(encoding="utf-8"))
summary_path = OUTPUT_ROOT / "post_dedup_results.md"
lines = ["# Post-Deduplication Model Results", "", f"Test images: {sum(after_counts['test'].values())}", "", "| Model | Accuracy | Macro-F1 | Training time (sec) | Evaluation time (sec) |", "|---|---:|---:|---:|---:|"]
for row in comparison:
    name = row["Model"]
    lines.append(f"| {name} | {row['Accuracy']:.6f} | {row['Macro-F1']:.6f} | {training_times.get(name, row.get('Training Time')):.2f} | {evaluation_times.get(name, 0.0):.2f} |")
best = max(comparison, key=lambda row: (row["Macro-F1"], row["Accuracy"]))
lines.extend(["", f"Best post-dedup model: **{best['Model']}**.", "", f"Artifacts: `{results_dir}`"])
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))

In [ ]:
required_outputs = [
    OUTPUT_ROOT / "checkpoints" / f"best_{name}.pt" for name in MODELS
] + [
    results_dir / "comparison_table.json",
    results_dir / "confusion_matrices_all.png",
    results_dir / "loss_curves.png",
    summary_path,
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected outputs: {missing}")
print("Portable pipeline completed successfully.")
print("Results directory:", results_dir)
print("Summary:", summary_path)